# Task 2.2 — Ablação de kernels do ECGClassifierV3 (Stage 2)

Este notebook avalia o impacto do tamanho dos kernels na primeira camada convolucional do classificador de arritmias ECG em duas etapas.  
Aqui focamos no **Stage 2** (classificação entre S, V e F), mantendo a arquitetura CNN + BiLSTM e variando apenas a tripla de kernels da torre convolucional.

**Restrições:**
- Dados sintéticos/dummy com shape `(n, 500, 1)` — nenhum dado real de ECG é carregado.
- Treinamento rápido (máx. 5 epochs) apenas para comparação relativa das configurações.
- Stack aprovada: TensorFlow/Keras, pandas, scikit-learn, matplotlib, seaborn.

In [ ]:
import os
import time
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import f1_score
from tensorflow import keras

from src.models.architecture_v3 import build_model

# Reprodutibilidade
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style='whitegrid', context='notebook')
print('TensorFlow:', tf.__version__)

## 1. Geração de dados sintéticos de ECG

Como os sinais reais estão armazenados em `data/raw_*` e são sensíveis à LGPD, utilizamos dados dummy com as mesmas características estruturais: 500 amostras, 1 canal, 3 classes (S, V, F) para o Stage 2.

In [ ]:
N_SAMPLES = 2_000
SIGNAL_LEN = 500
N_CLASSES = 3
CLASS_NAMES = ['S', 'V', 'F']


def generate_dummy_ecg(n: int, length: int = SIGNAL_LEN, seed: int = SEED) -> np.ndarray:
    """Gera sinais ECG sintéticos com shape (n, length, 1)."""
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 2 * np.pi, length)
    signals = []
    for i in range(n):
        freq = rng.uniform(0.8, 1.5)
        phase = rng.uniform(0, 2 * np.pi)
        noise = rng.normal(0, 0.05, size=length)
        # Soma de senoides + ruído para simular ECG de baixa fidelidade
        signal = (
            np.sin(freq * t + phase)
            + 0.3 * np.sin(3 * freq * t + phase)
            + 0.1 * np.sin(5 * freq * t + phase)
            + noise
        )
        signals.append(signal)
    return np.asarray(signals, dtype=np.float32).reshape(n, length, 1)


def generate_labels(n: int, n_classes: int = N_CLASSES, seed: int = SEED) -> np.ndarray:
    """Gera rótulos inteiros balanceados e os converte para one-hot."""
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(n_classes), n // n_classes)
    # Completar possíveis amostras remanescentes
    remainder = n - labels.shape[0]
    if remainder > 0:
        labels = np.concatenate([labels, rng.integers(0, n_classes, size=remainder)])
    labels = rng.permutation(labels)
    return keras.utils.to_categorical(labels, num_classes=n_classes)


X = generate_dummy_ecg(N_SAMPLES)
y = generate_labels(N_SAMPLES)

split_idx = int(0.8 * N_SAMPLES)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print('X_train:', X_train.shape, '| y_train:', y_train.shape)
print('X_val:  ', X_val.shape, '| y_val:  ', y_val.shape)

## 2. Grid de kernels e treinamento

Avaliamos quatro combinações de kernels para a torre convolucional.  
Todos os demais hiperparâmetros permanecem fixados para isolar o efeito da forma do receptive field.

In [ ]:
KERNEL_CONFIGS: List[Tuple[int, int, int]] = [
    (7, 5, 3),
    (9, 5, 3),
    (7, 7, 3),
    (5, 3, 3),
]

BASE_CONFIG = {
    'cnn_filters': [32, 64, 96],
    'use_gru': False,
    'dense_units': [128, 64],
    'dropout': [0.5, 0.3],
    'learning_rate': 1e-3,
    'loss': 'categorical_crossentropy',
}

EPOCHS = 5
BATCH_SIZE = 64
INFERENCE_REPETITIONS = 100

results: List[dict] = []

In [ ]:
for kernels in KERNEL_CONFIGS:
    config = {**BASE_CONFIG, 'cnn_kernels': list(kernels)}
    config_id = '—'.join(str(k) for k in kernels)
    print(f'\n>>> Treinando configuração: {config_id}')

    # Constrói modelo Stage 2 (3 classes: S, V, F)
    model = build_model(stage=2, config=config)

    start_fit = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
    )
    fit_time = time.perf_counter() - start_fit

    # Predições para F1 por classe
    y_pred_proba = model.predict(X_val, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_val, axis=1)

    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0, labels=range(N_CLASSES))

    # Tempo médio de inferência (ms/batimento)
    dummy_input = X_val[:1]
    _ = model.predict(dummy_input, verbose=0)  # warm-up
    start_inf = time.perf_counter()
    for _ in range(INFERENCE_REPETITIONS):
        _ = model.predict(dummy_input, verbose=0)
    inference_ms = (time.perf_counter() - start_inf) / INFERENCE_REPETITIONS * 1000

    result = {
        'config_id': config_id,
        'kernels': kernels,
        'f1_macro': f1_macro,
        'fit_time_s': fit_time,
        'inference_ms': inference_ms,
        'val_loss': history.history.get('val_loss', [np.nan])[-1],
        'val_f1_macro_keras': history.history.get('val_f1_macro', [np.nan])[-1],
    }
    for name, value in zip(CLASS_NAMES, f1_per_class):
        result[f'f1_{name}'] = value

    results.append(result)
    summary = (
        f'    F1-macro: {f1_macro:.4f} | '
        f'inferência: {inference_ms:.3f} ms | '
        f'treino: {fit_time:.1f} s'
    )
    print(summary)

print('\nAblação concluída.')

## 3. Resultados agregados

Tabela ordenada por F1-macro descendente.

In [ ]:
df_results = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
df_results = df_results[
    ['config_id', 'kernels', 'f1_macro', 'f1_S', 'f1_V', 'f1_F',
     'inference_ms', 'fit_time_s', 'val_loss', 'val_f1_macro_keras']
]
try:
    display(df_results)
except NameError:
    print(df_results.to_string())

best_config = df_results.iloc[0]
best_id = best_config['config_id']
best_f1 = best_config['f1_macro']
print(f'\nMelhor configuração: {best_id} (F1-macro = {best_f1:.4f})')


## 4. Heatmap de F1 por classe × configuração

In [ ]:
f1_columns = ['f1_S', 'f1_V', 'f1_F']
heatmap_data = df_results.set_index('config_id')[f1_columns]

plt.figure(figsize=(8, 4.5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    cmap='YlGnBu',
    cbar_kws={'label': 'F1-score'},
    linewidths=0.5,
)
plt.title('F1-score por classe para cada configuração de kernels (Stage 2)')
plt.xlabel('Classe')
plt.ylabel('Configuração de kernels')
plt.tight_layout()
plt.show()

## 5. Seleção automática das top-2 configurações

As duas melhores configurações, segundo F1-macro, são selecionadas automaticamente para validação posterior no pipeline completo.

In [ ]:
top2 = df_results.head(2)
selected_configs = [
    {'kernels': row['kernels'], 'f1_macro': row['f1_macro']}
    for _, row in top2.iterrows()
]

print('Top-2 configurações selecionadas automaticamente:')
for idx, cfg in enumerate(selected_configs, start=1):
    print(f'{idx}. kernels={cfg["kernels"]}  →  F1-macro={cfg["f1_macro"]:.4f}')

# Objeto disponível para próximos notebooks/scripts
SELECTED_KERNELS = [cfg['kernels'] for cfg in selected_configs]
print('\nSELECTED_KERNELS =', SELECTED_KERNELS)

## Observações

- Os valores absolutos de F1 são esperados baixos em dados dummy; o objetivo deste notebook é a **comparação relativa** entre configurações de kernels.
- Para resultados definitivos, execute o grid com dados reais resampleados (500 Hz, normalizados) e GroupKFold por paciente, conforme as Regras de Ouro do projeto.
- O tempo de inferência foi medido em CPU; no target STM32F4 a latência será determinada pelo TFLM + CMSIS-NN e deve respeitar QG9 (< 200 ms/batimento).